Backtracking
============

**Author:** Rafael



## Repaso de Python



### Argumentos opcionales



Las funciones en Python pueden tener argumentos opcionales. Tales argumentos opcionales pueden tener valores por defecto.



In [1]:
def suma(sumando1, sumando2=4):
    return sumando1 + sumando2

suma(5, 20)

In [1]:
suma(20)

Sin embargo, debemos tener cuidado con argumentos opcionales de tipo mutable.



In [1]:
def add_item(item, cart=[]):
    cart.append(item)
    return cart

alice_cart = add_item("apple")
alice_cart

In [1]:
bob_cart = add_item("lemon")
bob_cart

In [1]:
carol_cart = add_item("guitar", ["violin", "drums"])
carol_cart

In [1]:
david_cart = add_item("banana")
david_cart

El problema pasa porque cuando no usamos el segundo argumento, la misma lista se reusa automáticamente. Podríamos usar en lugar una tupla.



In [1]:
def add_item(item, cart=()):
    cart = cart + (item,)
    return cart

alice_cart = add_item("apple")
alice_cart

In [1]:
bob_cart = add_item("lemon")
bob_cart

In [1]:
carol_cart = add_item("guitar", ("violin", "drums"))
carol_cart

Sin embargo, de esta manera estamos creando varias tuplas innecesariamente. Otra estrategia es usar `None`.



In [1]:
def add_item(item, cart=None):
    if cart is None:
        cart = []  # creates a new cart each time
    
    cart.append(item)
    return cart

alice_cart = add_item("apple")
bob_cart = add_item("banana")
carol_cart = add_item("guitar", ["violin", "drums"])

# alice_cart, bob_cart, carol_cart
alice_cart = add_item("lemon")
alice_cart

## Generadores



## Bactracking



### Generando sucesiones de ceros y unos



Primera versión. Queremos generar primero listas que tienen muchos unos pues éstas nos darán soluciones al problema de la mochila.



In [1]:
def knapsack(l, X=None):
    # Initialize the single list on the first call
    if X is None:
        X = []

    if l == 3:
        # The list is complete, yield a copy of the list
        yield X[:] 
    else:
        # Branch 1: 
        X.append(1)
        yield from knapsack(l+1, X)
        X.pop()  # <--- Backtrack. Undo the choice.

        # Branch 2: Try 0
        X.append(0)
        yield from knapsack(l+1, X)
        X.pop()  # <--- Backtrack. Undo the choice.

# Test the generator
gens = knapsack(0)
list(gens)

Queremos generalizar esta idea, mostrando en cada momento diversas opciones para continuar.



In [1]:
def knapsack(l, X=None):
    if X is None:
        X = []

    if l == 4:
        # The list is complete: yield the snapshot of the list
        yield X[:] 
    else:
        # Define the choices available at this step
        choices = ["a", "b"] 
        
        # Loop through each choice
        for choice in choices:
            X.append(choice)             # 1. Make the choice
            yield from knapsack(l+1, X)  # 2. Explore that path
            X.pop()                      # 3. Backtrack (undo the choice)

gens = knapsack(0)
list(gens)

#### Generación de permutaciones



Veamos ahora otro ejemplo: generar permutaciones. Por ejemplo, de ["a", "b", "c"], generar: ["a", "b", "c"], ["a", "c", "b"], ["b", "a", "c"], etc.



In [1]:
def generate_permutations(elements, X=None):
    if X is None:
        X = []

    choices = [item for item in elements if item not in X] 
        
    for choice in choices:
        X.append(choice)                           # 1. Make choice
        yield from generate_permutations(elements, X)  # 2. Explore
        X.pop() # 3. Undo choice

list(generate_permutations([1,2,3,4]))

In [1]:
import networkx as nx
g = nx.petersen_graph()
nx.draw(g)

#### Generación de caminos en gráficas



In [1]:
import networkx as nx

def generate_walks(graph, X=None):
    if X is None:
        X = []

    if len(X) == 3:
        yield X[:] 
    else:
        if len(X) == 0:
            choices = graph.nodes()
        else:
            choices = [item for item in graph[X[-1]] if not item in X]
        
        # The core backtracking loop remains exactly the same!
        for choice in choices:
            X.append(choice)                     # 1. Make choice
            yield from generate_walks(graph, X)  # 2. Explore
            X.pop()                              # 3. Undo choice

list(generate_walks(nx.path_graph(4)))

#### Generación de ciclos hamiltonianos



In [1]:
def hamiltonian_cycle(graph, X=None):
    if X is None:
        X = [list(graph.nodes())[0]]

    if len(X) == len(graph.nodes()) and graph.has_edge(X[0], X[-1]):
        yield X[:] 
    else:
        choices = [vertex for vertex in graph[X[-1]] if vertex not in X]
        
        # The core backtracking loop remains exactly the same!
        for choice in choices:
            X.append(choice)                     # 1. Make choice
            yield from hamiltonian_cycle(graph, X)  # 2. Explore
            X.pop()                              # 3. Undo choice

In [1]:
import networkx as nx

list(hamiltonian_cycle(nx.cycle_graph(4)))

Con esto, podemos definir una función que verifique la propiedad de ser hamiltoniana usando una *excepción*.



In [1]:
def is_hamiltonian_graph(graph):
    generator = hamiltonian_cycle(graph)
    try:
        cycle = next(generator)
        return cycle
    except StopIteration:
        return False
        
is_hamiltonian_graph(nx.cycle_graph(4)), is_hamiltonian_graph(nx.path_graph(4))

#### Generación de particiones



En este caso calculamos separadamente la lista de elecciones posibles en cada paso, cuidando de no repetir particiones (es decir, no queremos reportar [3, 1] y [1, 3]), por lo que cuidamos que las elecciones sean menores o iguales al último número escogido.



In [1]:
def get_valid_numbers(remaining, X):
    """
    Auxiliary function to calculate valid choices.
    Filters choices to strictly break symmetry.
    """
    # If X is empty, the maximum number we can pick is the remaining total.
    # Otherwise, we can never pick a number larger than the last one we picked!
    max_allowed = X[-1] if len(X) > 0 else remaining
    
    # We can only pick numbers up to the smaller of the two constraints
    limit = min(remaining, max_allowed)
    
    # Return a list of choices counting down (e.g., [3, 2, 1])
    # Counting down generates the "largest" partitions first, which looks nicer
    return list(range(limit, 0, -1))


def integer_partitions(remaining, X=None):
    if X is None:
        X = []

    if remaining == 0:
        yield X[:]
    else:
        choices = get_valid_numbers(remaining, X)
        
        for choice in choices:
            X.append(choice)                              # Make choice
            yield from integer_partitions(remaining - choice, X)  # Explore (subtract from remaining)
            X.pop()

list(integer_partitions(6))

#### El problema de las n damas



Primero repasemos `enumerate`:



In [1]:
enumerate(["apple", "banana", "pear"])

In [1]:
list(enumerate(["apple", "banana", "pear"]))

In [1]:
for i, fruit in enumerate(["apple", "banana", "pear"]):
    print(f"Fruit {i} is {fruit}")

Ahora definimos una función auxiliar para calcular las posibles opciones de completar un tablero.



In [1]:
def get_safe_choices(N, board):
    """Auxiliary function to calculate all valid columns for the current row."""
    safe_choices = []
    current_row = len(board)
    
    for col in range(N):
        is_safe = True
        for r, c in enumerate(board):
            # Check if the column or diagonal is under attack
            if col == c or abs(current_row - r) == abs(col - c):
                is_safe = False
                break # No need to check further squares if it is False
                
        if is_safe:
            safe_choices.append(col)
            
    return safe_choices


def solve_n_queens(N, board=None):
    if board is None:
        board = []

    # Base case
    if len(board) == N:
        yield board[:]
    else:
        choices = get_safe_choices(N, board)
        
        for choice in choices:
            board.append(choice)                 # Make choice
            yield from solve_n_queens(N, board)  # Explore
            board.pop()                          # Undo choice

list(solve_n_queens(5))